In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
import xgboost as xgb
import shap

# Carregar dados
train_identity = pd.read_csv('../data/raw/train_identity.csv')
train_transaction = pd.read_csv('../data/raw/train_transaction.csv')
test_identity = pd.read_csv('../data/raw/test_identity.csv')
test_transaction = pd.read_csv('../data/raw/test_transaction.csv')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')  
test.columns = test.columns.str.replace('-', '_')
print(test.columns)  

print(train_transaction.shape)
print(train_transaction.dtypes.value_counts())
print(train_transaction['isFraud'].value_counts(normalize=True))

print(train.head())

: 

In [2]:
cat_cols = ['ProductCD', 'card4', 'card6', 'M1','M2','M3','M4','M5',
            'M6','M7','M8','M9','DeviceType','id_12','id_15', 'id_16','id_23','id_27','id_28',
            'id_29', 'id_30', 'id_31', 'id_35','id_36','id_37', 'id_38','id_33','id_34'] 

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[cat_cols] = enc.fit_transform(train[cat_cols])
test[cat_cols] = enc.transform(test[cat_cols])


In [3]:
# Combinações de campos que identificam o mesmo utilizador
train['uid1'] = train['card1'].astype(str) + '_' + train['addr1'].astype(str)
train['uid2'] = train['card1'].astype(str) + '_' + train['card2'].astype(str)
train['uid3'] = train['card1'].astype(str) + '_' + train['P_emaildomain'].astype(str)

test['uid1'] = test['card1'].astype(str) + '_' + test['addr1'].astype(str)
test['uid2'] = test['card1'].astype(str) + '_' + test['card2'].astype(str)
test['uid3'] = test['card1'].astype(str) + '_' + test['P_emaildomain'].astype(str)


for col in ['card1', 'card2', 'uid1', 'uid2', 'P_emaildomain', 'DeviceInfo']:
    freq = train[col].value_counts(normalize=True)
    train[f'{col}_freq'] = train[col].map(freq).fillna(0)
    test[f'{col}_freq'] = test[col].map(freq).fillna(0)

/tmp/ipykernel_4148/3006671742.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['uid1'] = train['card1'].astype(str) + '_' + train['addr1'].astype(str)
/tmp/ipykernel_4148/3006671742.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['uid2'] = train['card1'].astype(str) + '_' + train['card2'].astype(str)
/tmp/ipykernel_4148/3006671742.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns

In [4]:
# Para cada utilizador, estatísticas das suas transações
agg_dict = {
    'TransactionAmt': ['mean', 'std', 'max', 'min'],
    'D1': ['mean', 'std'],
    'C1': ['mean', 'sum'],
}

uid_agg = train.groupby('uid1').agg(agg_dict)
uid_agg.columns = ['uid1_' + '_'.join(c) for c in uid_agg.columns]
uid_agg = uid_agg.reset_index()

train = train.merge(uid_agg, on='uid1', how='left')
test = test.merge(uid_agg, on='uid1', how='left')

In [5]:
# log
train['amt_log'] = np.log1p(train['TransactionAmt'])
test['amt_log'] = np.log1p(test['TransactionAmt'])
# round value?
train['amt_is_round'] = (train['TransactionAmt'] % 1 == 0).astype(int)
test['amt_is_round'] = (test['TransactionAmt'] % 1 == 0).astype(int)

# Value vs user mean
train['amt_vs_uid_mean'] = train['TransactionAmt'] / (train['uid1_TransactionAmt_mean'] + 1)
test['amt_vs_uid_mean'] = test['TransactionAmt'] / (test['uid1_TransactionAmt_mean'] + 1)

# Decimal part of the amount
train['amt_decimal'] = train['TransactionAmt'] - np.floor(train['TransactionAmt'])  
test['amt_decimal'] = test['TransactionAmt'] - np.floor(test['TransactionAmt'])



/tmp/ipykernel_4148/1186063738.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['amt_log'] = np.log1p(train['TransactionAmt'])
/tmp/ipykernel_4148/1186063738.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['amt_log'] = np.log1p(test['TransactionAmt'])
/tmp/ipykernel_4148/1186063738.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a d

In [6]:
# Extrair domínio de topo
train['P_email_suffix'] = train['P_emaildomain'].str.split('.').str[-1]  # com, net, org...
test['P_email_suffix'] = test['P_emaildomain'].str.split('.').str[-1]

# Comprador e destinatário têm o mesmo email?
train['same_email'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(int)
test['same_email'] = (test['P_emaildomain'] == test['R_emaildomain']).astype(int)

email_fraud_rate = train.groupby('P_emaildomain')['isFraud'].mean()
train['P_email_fraud_rate'] = train['P_emaildomain'].map(email_fraud_rate)
test['P_email_fraud_rate'] = test['P_emaildomain'].map(email_fraud_rate)

/tmp/ipykernel_4148/3510824108.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['P_email_suffix'] = train['P_emaildomain'].str.split('.').str[-1]  # com, net, org...
/tmp/ipykernel_4148/3510824108.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['P_email_suffix'] = test['P_emaildomain'].str.split('.').str[-1]
/tmp/ipykernel_4148/3510824108.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

In [ ]:
cat_cols_not_label = ['P_emaildomain', 'P_email_suffix', 'DeviceInfo', 'uid1', 'uid2', 'uid3', 'R_emaildomain']

for col in cat_cols_not_label:
    freq = train[col].value_counts(normalize=True)
    
    train[col] = train[col].map(freq)
    
    if col in test.columns:
        test[col] = test[col].map(freq)
        test[col].fillna(0)  # valores novos no test recebem 0

    train[col].fillna(0)
    



/tmp/ipykernel_26485/502713010.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  test[col].fillna(0, inplace=True)  # valores novos no test recebem 0
/tmp/ipykernel_26485/502713010.py:12: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace

In [7]:
train = train.sort_values('TransactionDT').reset_index(drop=True)

n = len(train)

train_set = train.iloc[:int(n * 0.70)]  # 70% — treino
cv_set    = train.iloc[int(n * 0.70):int(n * 0.85)]  # 15% — validação
test_set  = train.iloc[int(n * 0.85):]  # 15% — teste final

print(f'Train: {len(train_set):,}')
print(f'CV:    {len(cv_set):,}')
print(f'Test:  {len(test_set):,}')

Train: 413,378
CV:    88,581
Test:  88,581


In [8]:
y_train = train_set[['isFraud']].values.ravel()
x_train = train_set.drop(columns=['TransactionID', 'isFraud', 'TransactionDT']) 
y_test = test_set[['isFraud']].values.ravel()
x_test = test_set.drop(columns=['TransactionID','isFraud', 'TransactionDT'])
y_cv = cv_set[['isFraud']].values.ravel()
x_cv = cv_set.drop(columns=['TransactionID', 'isFraud', 'TransactionDT'])

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(scale_pos_weight)

27.434310083918007


In [ ]:
def lgb_model():
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=100,
        max_depth=3,
        subsample=0.8,
        reg_lambda=0.2,
        early_stopping_rounds=100,
        verbose=1
    )

    model.fit(
        x_train, y_train,
        eval_set=[(x_cv, y_cv)],  # early stopping no CV
    )

    # Avaliar
    cv_preds   = model.predict_proba(x_cv)[:, 1]
    test_preds = model.predict_proba(x_test)[:, 1]

    print(f'CV AUC:   {roc_auc_score(y_cv, cv_preds):.4f}')
    print(f'Test AUC: {roc_auc_score(y_test, test_preds):.4f}')
    
    return model


In [ ]:
def xgb_model():
    model = xgb.XGBClassifier(
        n_estimators=500,        # número máximo de árvores
        learning_rate=0.03,       # taxa de aprendizado
        max_depth=6,              # profundidade da árvore
        min_child_weight=5,     # Exige mais amostras por folha (evita memorizar fraudes raras)
        gamma=1,
        subsample=0.8,            # amostragem de linhas
        colsample_bytree=0.8,     # amostragem de features
        scale_pos_weight=scale_pos_weight,  # balanceamento
        eval_metric='auc',        # métrica de avaliação
        random_state=42,
        early_stopping_rounds=50
    )

    model.fit(
        x_train, y_train,
        eval_set=[(x_cv, y_cv)],
        verbose=10
    )

    cv_preds   = model.predict_proba(x_cv)[:, 1]
    test_preds = model.predict_proba(x_test)[:, 1]

    print(f'CV AUC:   {roc_auc_score(y_cv, cv_preds):.4f}')
    print(f'Test AUC: {roc_auc_score(y_test, test_preds):.4f}')
    
    return model

In [10]:
test_ids = test['TransactionID']
x_final_test = test.drop(columns=['TransactionID', 'TransactionDT'])

In [11]:
model = lgb_model()
test_preds = model.predict_proba(x_final_test)[:, 1]

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: P_emaildomain: str, R_emaildomain: str, DeviceInfo: str, uid1: str, uid2: str, uid3: str, P_email_suffix: object

In [ ]:
submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": test_preds
})
print(submission.head())
print(submission.tail())
print(len(submission))
submission.to_csv("../submissions/submission.csv", index=False)

   TransactionID   isFraud
0        3663549  0.000745
1        3663550  0.002926
2        3663551  0.001303
3        3663552  0.001353
4        3663553  0.002434
        TransactionID   isFraud
506686        4170235  0.004865
506687        4170236  0.002848
506688        4170237  0.006364
506689        4170238  0.006200
506690        4170239  0.004596
506691
